In [2]:
# =====================================================
# Import Libraries
# =====================================================

import os
import joblib
import pandas as pd
import psycopg2

from dotenv import load_dotenv

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report
)

# =====================================================
# Connect to PostgreSQL
# =====================================================

load_dotenv()

connection = psycopg2.connect(
    host=os.getenv("DB_HOST"),
    port=os.getenv("DB_PORT"),
    dbname=os.getenv("DB_NAME"),
    user=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
    sslmode="require"
)

# =====================================================
# Load Training Data
# =====================================================

query = """
SELECT *
FROM ml_training_data
ORDER BY match_date, match_id;
"""

df = pd.read_sql_query(query, connection)

connection.close()

df["match_date"] = pd.to_datetime(df["match_date"])

print("=" * 60)
print("Dataset Loaded Successfully")
print("=" * 60)
print(f"Rows    : {df.shape[0]}")
print(f"Columns : {df.shape[1]}")

display(df.head())

# =====================================================
# Feature Selection
# =====================================================

features = [
    "win_rate_diff",
    "attack_efficiency_diff",
    "attack_kills_diff",
    "serve_aces_diff",
    "serve_errors_diff",
    "serve_efficiency_diff",
    "reception_positive_diff",
    "reception_perfect_diff",
    "block_points_diff",
    "block_touches_diff",
    "digs_diff",
    "assists_diff",
    "points_diff",
    "break_points_diff"
]

X = df[features]
y = df["target"]

# =====================================================
# Train-Test Split
# Week 1 -> Training
# Week 2 -> Testing
# =====================================================

train = df[df["week"] == 1]
test = df[df["week"] == 2]

X_train = train[features]
y_train = train["target"]

X_test = test[features]
y_test = test["target"]

print("\nTraining Samples :", len(train))
print("Testing Samples  :", len(test))

# =====================================================
# Train Logistic Regression Model
# =====================================================

model = LogisticRegression(
    max_iter=10000,
    random_state=42
)

model.fit(X_train, y_train)

print("\nModel trained successfully!")

# =====================================================
# Save Model
# =====================================================

os.makedirs("../models", exist_ok=True)

joblib.dump(
    model,
    "../models/logistic_regression.pkl"
)

print("Model saved to ../models/logistic_regression.pkl")

# =====================================================
# Generate Predictions
# =====================================================

predictions = model.predict(X_test)
probabilities = model.predict_proba(X_test)

# =====================================================
# Model Evaluation
# =====================================================

accuracy = accuracy_score(y_test, predictions)

print("\n" + "=" * 60)
print("MODEL PERFORMANCE")
print("=" * 60)

print(f"\nAccuracy: {accuracy:.2%}")

print("\nConfusion Matrix")
print(confusion_matrix(y_test, predictions))

print("\nClassification Report")
print(classification_report(y_test, predictions))

# =====================================================
# Prediction Results
# =====================================================

prediction_results = pd.DataFrame({
    "Actual": y_test.values,
    "Predicted": predictions,
    "Probability_Team_A_Wins": probabilities[:, 1]
})

display(prediction_results)

# =====================================================
# Feature Importance
# =====================================================

coef_df = pd.DataFrame({
    "Feature": features,
    "Coefficient": model.coef_[0]
})

coef_df["Absolute"] = coef_df["Coefficient"].abs()

coef_df = (
    coef_df
    .sort_values("Absolute", ascending=False)
    .drop(columns="Absolute")
)

print("\nFeature Importance (Logistic Regression)")
display(coef_df)

# =====================================================
# Save Outputs
# =====================================================

os.makedirs("../outputs", exist_ok=True)

prediction_results.to_csv(
    "../outputs/logistic_regression_predictions.csv",
    index=False
)

coef_df.to_csv(
    "../outputs/logistic_regression_coefficients.csv",
    index=False
)

metrics = pd.DataFrame({
    "Model": ["Logistic Regression"],
    "Accuracy": [accuracy]
})

metrics.to_csv(
    "../outputs/logistic_regression_metrics.csv",
    index=False
)

print("\nOutputs saved successfully.")

Dataset Loaded Successfully
Rows    : 63
Columns : 30


C:\Users\aulve\AppData\Local\Temp\ipykernel_4480\4112274223.py:44: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, connection)


,match_id,week,match_date,team_a,team_b,target,team_a_previous_matches,team_b_previous_matches,team_a_previous_wins,team_b_previous_wins,...,serve_errors_diff,serve_efficiency_diff,reception_positive_diff,reception_perfect_diff,block_points_diff,block_touches_diff,digs_diff,assists_diff,points_diff,break_points_diff
0,10,1,2026-06-11,6,17,0,1,1,0,0,...,3.0,-0.19,0.13,-0.06,1.0,0.0,12.0,23.0,21.0,4.0
1,11,1,2026-06-11,15,13,1,1,1,1,1,...,9.0,-0.13,-0.01,-0.03,2.0,2.0,11.0,27.0,24.0,5.0
2,12,1,2026-06-12,4,10,1,1,1,0,0,...,8.0,-0.05,-0.02,0.01,3.0,8.0,12.0,3.0,8.0,8.0
3,13,1,2026-06-12,9,11,0,1,1,1,0,...,-4.0,-0.05,-0.01,0.01,-3.0,-2.0,-4.0,3.0,-6.0,-13.0
4,14,1,2026-06-12,3,2,1,1,1,1,1,...,-3.0,0.00,0.12,0.00,7.0,4.0,-8.0,7.0,10.0,1.0



Training Samples : 27
Testing Samples  : 36

Model trained successfully!
Model saved to ../models/logistic_regression.pkl

MODEL PERFORMANCE

Accuracy: 55.56%

Confusion Matrix
[[ 9 11]
 [ 5 11]]

Classification Report
              precision    recall  f1-score   support

           0       0.64      0.45      0.53        20
           1       0.50      0.69      0.58        16

    accuracy                           0.56        36
   macro avg       0.57      0.57      0.55        36
weighted avg       0.58      0.56      0.55        36



,Actual,Predicted,Probability_Team_A_Wins
0,0,1,0.700963
1,1,0,0.330245
2,0,0,0.022935
3,1,0,0.266484
4,1,1,0.674172
5,0,0,0.219056
6,1,1,0.642683
7,1,1,0.562702
8,1,1,0.808940
9,0,1,0.605110



Feature Importance (Logistic Regression)


,Feature,Coefficient
8,block_points_diff,-0.419431
6,reception_positive_diff,0.221500
13,break_points_diff,0.149744
0,win_rate_diff,0.142079
4,serve_errors_diff,-0.133692
1,attack_efficiency_diff,0.119419
3,serve_aces_diff,-0.119380
12,points_diff,0.108281
2,attack_kills_diff,-0.089709
5,serve_efficiency_diff,0.073968



Outputs saved successfully.
